In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

dataset = load_dataset("emotion")
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

def tokenize(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

dataset = dataset.map(tokenize, batched=True)

In [ ]:
small_train = dataset["train"].shuffle(seed=42).select(range(1000))
small_eval = dataset["validation"].shuffle(seed=42).select(range(500))


In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained("bert-base-cased", num_labels=6)


In [ ]:
import numpy as np
import evaluate

metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [ ]:
from transformers import TrainingArguments
training_args = TrainingArguments(
    output_dir="emotion_classifier",        # train sonrası model dosyalarının kaydedileceği klasör
    eval_strategy="no",                  # validation her epoch sonunda yapılacak
    save_strategy="no",                  # model kaydı her epoch sonunda yapılacak
    learning_rate=2e-5,                     # ogrenme hızı
    per_device_train_batch_size=8,          # her train adımında kullanılacak batch boyutu 
    per_device_eval_batch_size=8,           # değerlendirme sırasında kullanılacak batch boyutu (eval)
    num_train_epochs=1,                     # Toplam 1 epoch
    weight_decay=0.01,                      # ağırlık cürümesi (regularization için)
    push_to_hub=False,                      #cok yavas calıstı false yaptim, egitilen modeli hub a yüklemek icin.
    logging_steps=50,                       #her 50 adimda loglama yap
    fp16=True,                              # 16-bit ile hız ve bellek tasarrufu sağlar (benimki yavas calistigi icin kullandim)
    report_to="none"                        
)


In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train,
    eval_dataset=small_eval,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()


In [7]:
id2label = {i: label for i, label in enumerate(dataset["train"].features["label"].names)}
model.config.id2label = id2label
model.config.label2id = {v: k for k, v in id2label.items()}


In [8]:

trainer.save_model("final_finetuned_model")
tokenizer.save_pretrained("final_finetuned_model")

('final_finetuned_model\\tokenizer_config.json',
 'final_finetuned_model\\special_tokens_map.json',
 'final_finetuned_model\\vocab.txt',
 'final_finetuned_model\\added_tokens.json',
 'final_finetuned_model\\tokenizer.json')

In [9]:
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model="final_finetuned_model",
    tokenizer=tokenizer,
    id2label=id2label
)

Device set to use cuda:0


In [10]:
classifier = pipeline("text-classification", model="final_finetuned_model")
print(classifier("I am so happy today!"))

Device set to use cuda:0


[{'label': 'fear', 'score': 0.22699938714504242}]
